
<h1 id="2.3-%E7%BC%96%E5%86%99%E5%8F%AF%E7%BC%96%E8%AF%91%E6%89%A7%E8%A1%8C%E7%9A%84%E5%88%86%E6%9E%90%E7%A8%8B%E5%BA%8F-(I)">2.3 编写可编译执行的分析程序 (I)</h1><h3 id="1.-%E8%BF%90%E8%A1%8C%E7%8E%AF%E5%A2%83%E7%9A%84%E8%BD%AC%E5%8F%98%EF%BC%9A%E4%BB%8E%E2%80%9C%E8%A7%A3%E9%87%8A%E2%80%9D%E5%88%B0%E2%80%9C%E7%BC%96%E8%AF%91%E2%80%9D">1. 运行环境的转变：从“解释”到“编译”</h3><ul>
<li><code>.x mycode.C</code> 由 ROOT 的 Cling 即时编译并执行，适合交互探索；不是逐行解释器，也不意味着事件循环必然很慢。</li>
<li>独立可执行程序使用 C++ 编译器和 ROOT 库构建，便于检查头文件依赖、组织多个源文件及批量运行。ACLiC 的 <code>.L mycode.C+</code> 也可编译宏。</li>
</ul>
<h3 id="2.-%E5%B7%A5%E7%A8%8B%E7%9B%AE%E5%BD%95%E7%BB%84%E7%BB%87">2. 工程目录组织</h3><p>一个标准的 C++ 分析工程应当将不同功能的文件分目录存放：</p>
<ul>
<li><strong><code>./tracking/</code></strong>：主目录，存放 <code>main.cpp</code>（程序入口）和 <code>Makefile</code>（构建脚本）。</li>
<li><strong><code>./tracking/include/</code></strong>：存放头文件 (<code>.h</code>, <code>.hh</code>)。</li>
<li><strong><code>./tracking/src/</code></strong>：存放源文件 (<code>.C</code>, <code>.cpp</code>)。</li>
</ul>
<hr/>
<h3 id="3.-%E5%88%A9%E7%94%A8-MakeClass-%E7%94%9F%E6%88%90%E5%9F%BA%E7%A1%80%E6%A1%86%E6%9E%B6">3. 利用 MakeClass 生成基础框架</h3><p>首先，利用 ROOT 的自动化工具生成映射 TTree 的基类：</p>
<div class="highlight"><pre><span></span>root<span class="w"> </span>-l<span class="w"> </span>f8ppac001.root
<span class="o">[</span><span class="m">0</span><span class="o">]</span><span class="w"> </span>tree-&gt;MakeClass<span class="o">(</span><span class="s2">"tracking"</span><span class="o">)</span>
</pre></div>
<p>生成的 <code>tracking.h</code> 移至 <code>include/</code>，<code>tracking.C</code> 移至 <code>src/</code>。</p>
<hr/>
<h3 id="4.-%E4%B8%BB%E7%A8%8B%E5%BA%8F%EF%BC%9Amain.cpp">4. 主程序：main.cpp</h3><p><code>main</code> 函数是整个分析程序的入口，负责调度。</p>
<ul>
<li><strong>参数解析</strong>：使用 <code>argc</code> 和 <code>argv</code> 接收命令行参数（如运行号 <code>run_number</code>）。</li>
<li><strong>文件名自动化</strong>：利用 <code>TString::Form()</code> 动态生成输入 (<code>f8ppac%03d.root</code>) 和输出 (<code>out%03d.root</code>) 文件名。</li>
<li><strong>流程</strong>：<ol>
<li>打开输入 <code>TFile</code> 并获取 <code>TTree</code> 指针。</li>
<li>创建输出 <code>TFile</code> 和用于存储结果的 <code>TTree</code>。</li>
<li><strong>核心实例化</strong>：<code>tracking *tk = new tracking(ipt);</code> 将输入树挂载到类中。</li>
<li><strong>启动循环</strong>：调用 <code>tk-&gt;Loop(opt);</code> 开始处理数据。</li>
<li>完成事件循环后调用 <code>output.Write()</code>。离开作用域时文件对象自动关闭。</li>
</ol>
</li>
</ul>
<div class="highlight"><pre><code class="language-cpp">#include &lt;TFile.h&gt;
#include &lt;TTree.h&gt;
#include &lt;TString.h&gt;
#include &lt;cstdlib&gt;
#include &lt;iostream&gt;
#include "tracking.h"

int main(int argc, char** argv) {
    if (argc!=2 &amp;&amp; argc!=4) {
        std::cerr &lt;&lt; "Usage: ./tracking run [input_dir output_dir]\n";
        return 1;
    }
    int run = std::atoi(argv[1]);
    const char* inputDir = argc==4 ? argv[2] : "../..";
    const char* outputDir = argc==4 ? argv[3] : ".";
    TString inputName = Form("%s/f8ppac%03d.root",inputDir,run);
    TString outputName = Form("%s/out%03d.root",outputDir,run);
    TFile* input = TFile::Open(inputName);
    if (!input || input-&gt;IsZombie()) return 1;
    TTree* tin = input-&gt;Get&lt;TTree&gt;("tree");
    if (!tin) return 1;
    TFile output(outputName,"RECREATE");
    if (output.IsZombie()) return 1;
    TTree* tout = new TTree("tree","PPAC tracking");
    {
        tracking analysis(tin);
        analysis.Loop(tout);

        std::cout &lt;&lt; "Input=" &lt;&lt; tin-&gt;GetEntries() &lt;&lt; ", output=" &lt;&lt; tout-&gt;GetEntries() &lt;&lt; '\n';
        output.Write();
    } // MakeClass 基类析构时释放输入文件；不再重复 delete input。
    return 0;
}</code></pre></div>
<hr/>
<p>在编写复杂的 C++ 程序时，遵循“<strong>声明与实现分离</strong>”的原则至关重要：</p>
<ul>
<li><strong>头文件 (.h)</strong>：只写类的声明、成员变量、函数原型和宏定义。本例将实现放入源文件；inline 与模板可以在头文件定义。</li>
<li><strong>源文件 (.C / .cpp)</strong>：” 编写函数的具体逻辑。</li>
</ul>
<p>这种分离可以加快编译速度，并使代码逻辑更加清晰。</p>
<h3 id="5.-%E5%A4%B4%E6%96%87%E4%BB%B6%EF%BC%9Ainclude/tracking.h">5. 头文件：include/tracking.h</h3><p>头文件用于声明类、变量和方法。</p>
<p>在编写 <code>./include/tracking.h</code> 时，必须在文件的<strong>开头</strong>和<strong>结尾</strong>加上预编译语句。这是为了防止在复杂的工程中，同一个头文件被多次 <code>#include</code> 而导致重复定义错误。</p>
<p><strong>规范格式如下：</strong></p>
<div class="highlight"><pre><span></span><span class="cp">#ifndef TRACKING_H   </span><span class="c1">// 如果没有定义 TRACKING_H</span>
<span class="cp">#define TRACKING_H   </span><span class="c1">// 那么定义 TRACKING_H</span>

<span class="c1">// --- 你的代码声明写在这里 ---</span>
<span class="k">class</span><span class="w"> </span><span class="nc">tracking</span><span class="w"> </span><span class="p">{</span>
<span class="w">    </span><span class="c1">// ...</span>
<span class="p">};</span>

<span class="cp">#endif               </span><span class="c1">// 结束判断</span>
</pre></div>
<ul>
<li>通常使用大写的文件名（如 <code>TRACKING_H</code>），只要确保在整个工程中唯一即可。</li>
</ul>
<div class="highlight"><pre><code class="language-cpp">//////////////////////////////////////////////////////////
// This class has been automatically generated on
// Sun Sep 13 09:54:23 2026 by ROOT version 6.40.02
// from TTree tree/tree
// found on file: /Users/zhli/Desktop/course/code-new/data/chapt2/f8ppac001.root
//////////////////////////////////////////////////////////

#ifndef tracking_h
#define tracking_h

#include &lt;TROOT.h&gt;
#include &lt;TH2.h&gt;
#include &lt;TChain.h&gt;
#include &lt;TFile.h&gt;

// Header file for the classes stored in the TTree if any.

class tracking {
public :
   Double_t xx[3], xz[3], yy[3], yz[3], dx[3], dy[3];
   Double_t xx2b[2], yy2b[2], xz2b, yz2b, anode2b;
   Double_t tx,ty,theta_x,theta_y,sigma_tx,sigma_ty,sigma_thetax,sigma_thetay,c2nx,c2ny;
   Long64_t source_entry;
   void SetBranch(TTree *tree);
   void TrackInit();
   void SetTrace(TH2D *h, Double_t k, Double_t b, Int_t min, Int_t max);

   TTree          *fChain;   ///&lt;!pointer to the analyzed TTree or TChain
   Int_t           fCurrent; ///&lt;!current Tree number in a TChain

// Fixed size dimensions of array or collections stored in the TTree if any.

   // Declaration of leaf types
   Float_t         PPACF8[5][5];
   Float_t         F8PPACRawData[5][5];
   Int_t           beamTrig;
   Int_t           must2Trig;
   Float_t         targetX;
   Float_t         targetY;

   // List of branches
   TBranch        *b_PPACF8;   ///&lt;!
   TBranch        *b_F8PPACRawData;   ///&lt;!
   TBranch        *b_beamTrig;   ///&lt;!
   TBranch        *b_must2Trig;   ///&lt;!
   TBranch        *b_targetX;   ///&lt;!
   TBranch        *b_targetY;   ///&lt;!

   tracking(TTree *tree=0);
   virtual ~tracking();
   virtual Int_t    Cut(Long64_t entry);
   virtual Int_t    GetEntry(Long64_t entry);
   virtual Long64_t LoadTree(Long64_t entry);
   virtual void     Init(TTree *tree);
   virtual void     Loop(TTree *tree);
   virtual bool     Notify();
   virtual void     Show(Long64_t entry = -1);
};

#endif</code></pre></div>
<hr/>
<h3 id="6.-%E6%BA%90%E6%96%87%E4%BB%B6%EF%BC%9Asrc/tracking.C">6. 源文件：src/tracking.C</h3><p>在源文件中，我们通过 <code>tracking::</code> 前缀来实现头文件中声明的方法。</p>
<p><strong>注意：必须包含 <code>#include "tracking.h"</code>。</strong></p>
<div class="highlight"><pre><code class="language-cpp">#define tracking_cxx
#include "tracking.h"
#include &lt;TH2.h&gt;
#include &lt;TStyle.h&gt;
#include &lt;TCanvas.h&gt;
#include &lt;TF1.h&gt;
#include &lt;TGraphErrors.h&gt;   // 【修改】必须包含带有误差的图类
#include &lt;TFitResult.h&gt;
#include &lt;TMatrixDSym.h&gt;    // 【新增】用于接收协方差矩阵
#include &lt;iostream&gt;
#include &lt;cmath&gt;

using namespace std;

void tracking::SetBranch(TTree *tree)
{
    tree-&gt;Branch("source_entry", &amp;source_entry, "source_entry/L");
    tree-&gt;Branch("xx", xx, "xx[3]/D");
    tree-&gt;Branch("xz", xz, "xz[3]/D");
    tree-&gt;Branch("yy", yy, "yy[3]/D");
    tree-&gt;Branch("yz", yz, "yz[3]/D");
    tree-&gt;Branch("dx", dx, "dx[3]/D");
    tree-&gt;Branch("dy", dy, "dy[3]/D");
    tree-&gt;Branch("xx2b", xx2b, "xx2b[2]/D");
    tree-&gt;Branch("yy2b", yy2b, "yy2b[2]/D");
    tree-&gt;Branch("anode2b", &amp;anode2b, "anode2b/D");

    // 【新增】注册所有运动学中心值及其物理误差
    tree-&gt;Branch("tx", &amp;tx, "tx/D");
    tree-&gt;Branch("ty", &amp;ty, "ty/D");
    tree-&gt;Branch("theta_x", &amp;theta_x, "theta_x/D");
    tree-&gt;Branch("theta_y", &amp;theta_y, "theta_y/D");
    tree-&gt;Branch("sigma_tx", &amp;sigma_tx, "sigma_tx/D");
    tree-&gt;Branch("sigma_ty", &amp;sigma_ty, "sigma_ty/D");
    tree-&gt;Branch("sigma_thetax", &amp;sigma_thetax, "sigma_thetax/D");
    tree-&gt;Branch("sigma_thetay", &amp;sigma_thetay, "sigma_thetay/D");

    tree-&gt;Branch("c2nx", &amp;c2nx, "c2nx/D");
    tree-&gt;Branch("c2ny", &amp;c2ny, "c2ny/D");
    tree-&gt;Branch("beamTrig", &amp;beamTrig, "beamTrig/I");
    tree-&gt;Branch("must2Trig", &amp;must2Trig, "must2Trig/I");
    tree-&gt;Branch("targetX", &amp;targetX, "targetX/F");
    tree-&gt;Branch("targetY", &amp;targetY, "targetY/F");
}

void tracking::TrackInit()
{
    // 初始化所有计算变量为无效值，防止上一个事件的数据污染
    tx = -999; ty = -999;
    c2nx = -1; c2ny = -1;
    for (int i=0;i&lt;3;++i) { dx[i]=-999; dy[i]=-999; }
    theta_x = -999; theta_y = -999;
    sigma_tx = -1; sigma_ty = -1;             // 误差初始化为负数代表无效
    sigma_thetax = -1; sigma_thetay = -1;

    xx[0] = PPACF8[0][0];  yy[0] = PPACF8[0][1];  xz[0] = PPACF8[0][2];  yz[0] = PPACF8[0][3];
    xx[1] = PPACF8[2][0];  yy[1] = PPACF8[2][1];  xz[1] = PPACF8[2][2];  yz[1] = PPACF8[2][3];
    xx[2] = PPACF8[4][0];  yy[2] = PPACF8[4][1];  xz[2] = PPACF8[4][2];  yz[2] = PPACF8[4][3];

    xx2b[0] = PPACF8[3][0]; yy2b[0] = PPACF8[3][1];
    xz2b    = PPACF8[3][2]; yz2b    = PPACF8[3][3];
    anode2b = PPACF8[3][4];

    xx2b[1] = -1000; yy2b[1] = -1000;
}

void tracking::SetTrace(TH2D *h, Double_t k, Double_t b, Int_t min, Int_t max){
    if(h == 0 || min &gt;= max) return;
    for(int i = min; i &lt; max; i++){
        h-&gt;Fill(i, i * k + b);
    }
}

void tracking::Loop(TTree *tree)
{
    if (fChain == 0) return;

    SetBranch(tree);

    TH2D *htf8xz = new TH2D("htf8xz", "X-Z Plane Trace; Z (mm); X (mm)", 2200, -2000, 200, 300, -150, 150);
    TH2D *htf8yz = new TH2D("htf8yz", "Y-Z Plane Trace; Z (mm); Y (mm)", 2200, -2000, 200, 300, -150, 150);

    // 【核心修改】使用 TGraphErrors 替代 TGraph，输入假设的单层位置误差
    TGraphErrors *grx = new TGraphErrors(3);
    TGraphErrors *gry = new TGraphErrors(3);
    TF1 *fx = new TF1("fx", "pol1", -2000, 0);
    TF1 *fy = new TF1("fy", "pol1", -2000, 0);

    // 假设：所有PPAC每层的本征位置分辨率为 1.0 mm
    const double det_resolution = 1.0;
    const double z_target = 0.0; // 物理靶所在Z坐标位置

    Long64_t nentries = fChain-&gt;GetEntriesFast();
    Long64_t nbytes = 0, nb = 0;

    for (Long64_t jentry = 0; jentry &lt; nentries; jentry++) {
        Long64_t ientry = LoadTree(jentry);
        if (ientry &lt; 0) break;
        nb = fChain-&gt;GetEntry(jentry);   nbytes += nb;

        source_entry = jentry;
        TrackInit();

        bool b1a = abs(xx[0]) &lt; 150 &amp;&amp; abs(yy[0]) &lt; 150;
        bool b2a = abs(xx[1]) &lt; 150 &amp;&amp; abs(yy[1]) &lt; 150;
        bool b3  = abs(xx[2]) &lt; 100 &amp;&amp; abs(yy[2]) &lt; 100;
        if(!b1a || !b2a || !b3) continue;

        // ================= X-Z 平面径迹拟合与误差计算 =================
        for(int i=0; i&lt;3; i++) {
            grx-&gt;SetPoint(i, xz[i], xx[i]);
            grx-&gt;SetPointError(i, 0.0, det_resolution); // 关键：输入Z和X的误差
        }

        // 【核心修改】去除 "W" 选项。S=保存结果(以获取矩阵), Q=静默模式
        TFitResultPtr rx = grx-&gt;Fit(fx, "SQN");

        if (int(rx)==0 &amp;&amp; rx.Get() &amp;&amp; rx-&gt;IsValid()) {
            double p0_x = fx-&gt;GetParameter(0);
            double p1_x = fx-&gt;GetParameter(1);

            // 提取中心值
            xx2b[1] = fx-&gt;Eval(xz2b);
            tx      = p0_x + p1_x * z_target;
            theta_x = atan(p1_x); // 物理出射角 (rad)

            // 提取协方差矩阵并计算严谨物理误差
            TMatrixDSym cov_x = rx-&gt;GetCovarianceMatrix();
            double var_p0 = cov_x(0, 0);
            double var_p1 = cov_x(1, 1);
            double cov_p0_p1 = cov_x(0, 1);

            // 外推位置误差传递公式
            double err2_tx = var_p0 + (z_target * z_target * var_p1) + (2.0 * z_target * cov_p0_p1);
            sigma_tx = sqrt(err2_tx);

            // 角度非线性误差传递公式
            sigma_thetax = sqrt(var_p1) / (1.0 + p1_x * p1_x);

            c2nx = rx-&gt;Chi2() / rx-&gt;Ndf();
            if (jentry &lt; 10000) SetTrace(htf8xz, p1_x, p0_x, -1800, 0);
            for(int i=0; i&lt;3; i++) dx[i] = xx[i] - fx-&gt;Eval(xz[i]);
        }

        // ================= Y-Z 平面径迹拟合与误差计算 =================
        for(int i=0; i&lt;3; i++) {
            gry-&gt;SetPoint(i, yz[i], yy[i]);
            gry-&gt;SetPointError(i, 0.0, det_resolution);
        }

        TFitResultPtr ry = gry-&gt;Fit(fy, "SQN");

        if (int(ry)==0 &amp;&amp; ry.Get() &amp;&amp; ry-&gt;IsValid()) {
            double p0_y = fy-&gt;GetParameter(0);
            double p1_y = fy-&gt;GetParameter(1);

            yy2b[1] = fy-&gt;Eval(yz2b);
            ty      = p0_y + p1_y * z_target;
            theta_y = atan(p1_y);

            TMatrixDSym cov_y = ry-&gt;GetCovarianceMatrix();
            double var_p0 = cov_y(0, 0);
            double var_p1 = cov_y(1, 1);
            double cov_p0_p1 = cov_y(0, 1);

            double err2_ty = var_p0 + (z_target * z_target * var_p1) + (2.0 * z_target * cov_p0_p1);
            sigma_ty = sqrt(err2_ty);

            sigma_thetay = sqrt(var_p1) / (1.0 + p1_y * p1_y);

            c2ny = ry-&gt;Chi2() / ry-&gt;Ndf();
            if (jentry &lt; 10000) SetTrace(htf8yz, p1_y, p0_y, -1800, 0);
            for(int i=0; i&lt;3; i++) dy[i] = yy[i] - fy-&gt;Eval(yz[i]);
        }

        // 将本事件结果写入 Tree (包括新算出的误差)
        if (c2nx&gt;=0 &amp;&amp; c2ny&gt;=0) tree-&gt;Fill();

        if(jentry % 10000 == 0) cout &lt;&lt; "Processing Event: " &lt;&lt; jentry &lt;&lt; " / " &lt;&lt; nentries &lt;&lt; endl;
    }

    // 释放内存并保存结果
    delete grx; delete gry;
    delete fx;  delete fy;




    cout &lt;&lt; "Input events=" &lt;&lt; nentries &lt;&lt; ", accepted reference tracks=" &lt;&lt; tree-&gt;GetEntries() &lt;&lt; endl;



}</code></pre></div>
<hr/>
<h3 id="7.-%E8%87%AA%E5%8A%A8%E5%8C%96%E7%BC%96%E8%AF%91%EF%BC%9AMakefile">7. 自动化编译：Makefile</h3><p>为了将上述分散在不同目录下的代码（主程序、源文件、头文件）一键编译为可执行程序，需要利用 <code>Makefile</code> 进行自动化构建。</p>
<p><code>Makefile</code> 相当于一个构建剧本，告诉 <code>make</code> 命令如何正确地传递参数、编译源代码并链接所需的动态库。</p>
<ul>
<li><strong>获取 ROOT 环境参数</strong>：通过 ROOT 提供的自带工具 <code>root-config</code>，程序可以自动适配不同电脑上的 ROOT 安装路径。<ul>
<li><code>ROOTCFLAGS = $(shell root-config --cflags)</code>：获取 ROOT 的头文件搜索路径和预处理宏。</li>
<li><code>ROOTLIBS = $(shell root-config --libs)</code>：获取 ROOT 的核心基础架构库（Core, Tree, Hist 等）。</li>
<li><code>ROOTGLIBS = $(shell root-config --glibs)</code>：获取包含图形界面支持的更全面的库（GUI, Gpad 等）。</li>
</ul>
</li>
<li>编译与链接指令：<code>$(CXX) $(CPPFLAGS) $(CXXFLAGS) $(SOURCES) $(LDLIBS) -o $@</code>。这里把编译和链接合并为一步，<code>$@</code> 是目标文件名 tracking。</li>
</ul>
<p><strong>以下为标准 <code>Makefile</code> 脚本：</strong></p>
<div class="highlight"><pre><code class="language-cpp">CXX = c++
CPPFLAGS = -Iinclude $(shell root-config --cflags)
CXXFLAGS = -O2 -Wall
LDLIBS = $(shell root-config --libs)
SOURCES = main.cpp $(wildcard src/*.cpp src/*.C)
HEADERS = $(wildcard include/*.h)

all: tracking

tracking: $(SOURCES) $(HEADERS)
	$(CXX) $(CPPFLAGS) $(CXXFLAGS) $(SOURCES) $(LDLIBS) -o $@

clean:
	rm -f tracking</code></pre></div>
<h4 id="Makefile-%E5%85%B3%E9%94%AE%E7%82%B9%E8%A7%A3%E6%9E%90%EF%BC%9A">Makefile 关键点解析：</h4><ol>
<p><code>$(wildcard src/*.cpp src/*.C)</code> 列出源文件；<code>$(wildcard include/*.h)</code> 列出头文件，作为重新构建的依赖。</p>
<p><code>-Iinclude</code> 指定头文件搜索目录，因此源文件中可以写 <code>#include "tracking.h"</code>。</p>
<li>本例将编译选项放在 CPPFLAGS/CXXFLAGS，将链接库放在 LDLIBS。库通常放在源文件或目标文件之后，避免静态库符号解析的顺序问题；变量名本身没有特殊语法含义。</li>
</ol>
<hr/>
<p>完整工程位于 <code>code/compile1</code>。Loop 沿用 2.2 的同一套 tracking 和误差传播；main 负责文件，Loop 负责填充传入的树。头文件中的生成代码仍保留，不应从上面的节选中删去构造函数和分支绑定。</p>

<h3>编译和运行</h3><p>在 <code>code/compile1</code> 中执行：</p><pre><code class="language-cpp">make
./tracking 1
# 或显式指定输入、输出目录
./tracking 1 ../.. .</code></pre><p><code>root-config --cflags</code> 给编译选项，<code>--libs</code> 给链接选项。源文件中包含声明所用类型的头文件（如 TGraphErrors.h）；这与链接 ROOT 库是两件不同的事。编译报错时先检查第一处错误。</p>

In [3]:
!make -C code/compile1

make: Nothing to be done for `all'.

In [4]:
!cd code/compile1 && ./tracking 1

Processing Event: 30000 / 739685
Processing Event: 50000 / 739685
Processing Event: 110000 / 739685
Processing Event: 130000 / 739685
Processing Event: 140000 / 739685
Processing Event: 150000 / 739685
Processing Event: 190000 / 739685
Processing Event: 240000 / 739685
Processing Event: 270000 / 739685
Processing Event: 360000 / 739685
Processing Event: 390000 / 739685
Processing Event: 400000 / 739685
Processing Event: 410000 / 739685
Processing Event: 440000 / 739685
Processing Event: 450000 / 739685
Processing Event: 490000 / 739685
Processing Event: 550000 / 739685
Processing Event: 560000 / 739685
Processing Event: 590000 / 739685
Processing Event: 600000 / 739685
Processing Event: 620000 / 739685
Processing Event: 630000 / 739685
Processing Event: 640000 / 739685
Processing Event: 650000 / 739685
Input events=739685, accepted reference tracks=232180
Input=739685, output=232180